# BiasLens Worker ML Notebook

This notebook mirrors the ML module so Google Colab can discover and run it directly from GitHub.

Flow:
1. Install dependencies
2. Write `worker_ml.py` to the Colab runtime
3. Import and validate the module
4. Confirm runtime readiness for `worker_service.py`

In [ ]:
# Cell 1: Install ML dependencies
!pip -q install \
  numpy==2.2.6 \
  pandas==2.3.0 \
  torch==2.5.1 \
  torchvision==0.20.1 \
  opencv-python==4.11.0.86 \
  Pillow==11.3.0 \
  boto3==1.39.13

import torch
print("Dependencies installed")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: Create worker_ml.py in Colab runtime

worker_ml_source = '''"""
Heavy-lifting ML module for BiasLens Colab Worker.
Handles PyTorch model training, Grad-CAM generation, and subgroup analysis.
Designed for GPU-accelerated Colab environments.
"""

import io
import tempfile
from collections import Counter
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import efficientnet_b0

import boto3


class ImageDataset(Dataset):
    """PyTorch Dataset for medical images with binary classification labels."""

    def __init__(self, image_paths: list[str], labels: list[int], transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        try:
            img = Image.open(img_path).convert("RGB")
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception as e:
            print(f"Warning: Failed to load {img_path}: {e}")
            black = Image.new("RGB", (224, 224))
            if self.transform:
                black = self.transform(black)
            return black, label


class GradCAMGenerator:
    """Generates Grad-CAM heatmaps for identifying shortcut patterns in images."""

    def __init__(self, model: nn.Module, target_layer: str = "features.8"):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        target = dict(self.model.named_modules())[self.target_layer]
        target.register_forward_hook(forward_hook)
        target.register_backward_hook(backward_hook)

    def generate(self, image_tensor: torch.Tensor, class_idx: int = None) -> np.ndarray:
        self.model.eval()
        image_tensor = image_tensor.unsqueeze(0)

        with torch.enable_grad():
            output = self.model(image_tensor)
            if class_idx is None:
                class_idx = output.argmax(dim=1).item()

            self.model.zero_grad()
            score = output[0, class_idx]
            score.backward()

        if self.gradients is None or self.activations is None:
            return np.zeros((224, 224), dtype=np.uint8)

        gradients = self.gradients[0].cpu().numpy()
        activations = self.activations[0].cpu().numpy()
        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)

        for w, act in zip(weights, activations):
            cam += w * act

        cam = np.maximum(cam, 0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam, (224, 224))
        cam = (cam * 255).astype(np.uint8)

        return cam


def train_proxy_model(
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 5,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> tuple[nn.Module, dict[str, float]]:
    model = efficientnet_b0(weights="DEFAULT")
    model.classifier[1] = nn.Linear(1280, 2)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_val_acc = 0.0
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100 * train_correct / train_total

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total
        scheduler.step(val_acc)

        print(
            f"Epoch [{epoch + 1}/{epochs}] Train Loss: {train_loss / len(train_loader):.4f}, "
            f"Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    return model, {"validation_accuracy": best_val_acc, "final_train_accuracy": train_acc}


def analyze_subgroups(
    df: pd.DataFrame,
    image_paths: list[str],
    model: nn.Module,
    image_to_label: dict[str, int],
    demographic_col: str = None,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> dict[str, Any]:
    subgroup_results = {}

    if not demographic_col:
        demographic_col = _pick_source_column(df)
    if not demographic_col:
        return {"subgroups": [], "disparity_detected": False}

    model.eval()
    transform = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    subgroup_accuracies = {}
    for subgroup in df[demographic_col].unique():
        mask = df[demographic_col] == subgroup
        subgroup_indices = mask[mask].index.tolist()

        if not subgroup_indices:
            continue

        correct = 0
        total = 0
        for idx in subgroup_indices:
            if idx < len(image_paths):
                img_path = image_paths[idx]
                true_label = image_to_label.get(img_path, 0)

                try:
                    img = Image.open(img_path).convert("RGB")
                    img = transform(img).unsqueeze(0).to(device)

                    with torch.no_grad():
                        output = model(img)
                        pred_label = output.argmax(dim=1).item()

                    if pred_label == true_label:
                        correct += 1
                    total += 1
                except Exception as e:
                    print(f"Error processing image for subgroup analysis: {e}")
                    total += 1

        if total > 0:
            accuracy = 100 * correct / total
            subgroup_accuracies[str(subgroup)] = accuracy

    if subgroup_accuracies:
        max_acc = max(subgroup_accuracies.values())
        min_acc = min(subgroup_accuracies.values())
        disparity = max_acc - min_acc

        subgroup_results = {
            "subgroups": [
                {"name": name, "accuracy": acc, "sample_count": int((df[demographic_col] == name).sum())}
                for name, acc in subgroup_accuracies.items()
            ],
            "max_accuracy": max_acc,
            "min_accuracy": min_acc,
            "disparity_percent": disparity,
            "disparity_detected": disparity > 15,
        }
    else:
        subgroup_results = {
            "subgroups": [],
            "max_accuracy": 0,
            "min_accuracy": 0,
            "disparity_percent": 0,
            "disparity_detected": False,
        }

    return subgroup_results


def generate_heatmap_collage(
    image_paths: list[str],
    model: nn.Module,
    num_samples: int = 6,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> bytes:
    model.eval()
    gradcam = GradCAMGenerator(model)

    transform = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    collage_height = 224 * 2
    collage_width = 224 * 3
    collage = np.ones((collage_height, collage_width, 3), dtype=np.uint8) * 255

    sample_paths = image_paths[:num_samples]

    for idx, img_path in enumerate(sample_paths):
        row = idx // 3
        col = idx % 3
        y_offset = row * 224
        x_offset = col * 224

        try:
            img = Image.open(img_path).convert("RGB")
            img_array = np.array(img.resize((224, 224)))

            img_tensor = transform(img).to(device)
            heatmap = gradcam.generate(img_tensor)

            overlay = np.zeros_like(img_array)
            overlay[:, :, 0] = heatmap
            overlay[:, :, 1:] = img_array[:, :, 1:] * 0.5

            blended = cv2.addWeighted(img_array, 0.6, overlay, 0.4, 0)
            collage[y_offset : y_offset + 224, x_offset : x_offset + 224] = blended
        except Exception as e:
            print(f"Warning: Could not process {img_path} for collage: {e}")

    collage_img = Image.fromarray(collage)
    png_bytes = io.BytesIO()
    collage_img.save(png_bytes, format="PNG")
    return png_bytes.getvalue()


def upload_heatmap_to_s3(
    heatmap_bytes: bytes,
    bucket: str,
    key: str,
    aws_access_key_id: str = None,
    aws_secret_access_key: str = None,
    aws_session_token: str = None,
    region_name: str = "us-east-1",
) -> str:
    session_kwargs = {}
    if aws_access_key_id:
        session_kwargs["aws_access_key_id"] = aws_access_key_id
    if aws_secret_access_key:
        session_kwargs["aws_secret_access_key"] = aws_secret_access_key
    if aws_session_token:
        session_kwargs["aws_session_token"] = aws_session_token

    s3 = boto3.client("s3", region_name=region_name, **session_kwargs)
    s3.put_object(Bucket=bucket, Key=key, Body=heatmap_bytes, ContentType="image/png")

    return f"s3://{bucket}/{key}"


def _pick_source_column(df: pd.DataFrame) -> str | None:
    preferred = ["source", "site", "hospital", "domain", "scanner", "institution"]
    lower_map = {str(col).lower(): col for col in df.columns}

    for key in preferred:
        if key in lower_map:
            return str(lower_map[key])

    return None
'''

with open("worker_ml.py", "w", encoding="utf-8") as f:
    f.write(worker_ml_source)

print("Created worker_ml.py in runtime")

In [ ]:
# Cell 3: Import and smoke-test worker_ml module

import importlib

worker_ml = importlib.import_module("worker_ml")
print("Imported worker_ml successfully")
print("Has ImageDataset:", hasattr(worker_ml, "ImageDataset"))
print("Has GradCAMGenerator:", hasattr(worker_ml, "GradCAMGenerator"))
print("Has train_proxy_model:", hasattr(worker_ml, "train_proxy_model"))

In [ ]:
# Cell 4: Optional service check in same Colab runtime
# This confirms both ML and service modules can coexist in one notebook instance.

try:
    import worker_service
    print("worker_service imported successfully")
    print("You can now run worker_service.start_worker() in this same runtime.")
except Exception as e:
    print("worker_service import failed:", e)
    print("If needed, copy worker_service.py into the same Colab runtime or clone the repo first.")

In [ ]:
# Cell 5: Optional backend live ping (health notification)

import os
import requests

backend_health_url = os.getenv("BACKEND_HEALTH_URL", "")

if backend_health_url:
    try:
        r = requests.get(backend_health_url, timeout=10)
        print("Backend health ping status:", r.status_code)
        print("Response:", r.text[:200])
    except Exception as e:
        print("Failed backend health ping:", e)
else:
    print("Set BACKEND_HEALTH_URL env var to notify/check backend, for example:")
    print("https://your-backend-domain/api/v1/health")